**Author**: Felipe Matheus
**Purpose**: Experiment launcher for the annealing **tensile strength (UTS)** surrogate.

Same architecture as `run_experiments.ipynb` (IACS): all pipeline logic lives
in `src/modeling/Experiments.py`, which is process-agnostic — the SAME
`ExperimentRunner` is reused; only the `ExperimentConfig` changes (target,
features, physical bounds). No new class needed.

Results layout: `models/annealing_tensile_strength/experiments/`
(`experiments_log.csv` + one folder per run).

# 1. Setup

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
TARGET = "tensile_strength_final"
ALL_FEATURES = ["purity", "initial_diameter", "tensile_strength", "temperature", "time"]
PROCESS = "annealing_uts"

TAG = "annealing-uts-v1-best_quality"
GRID = {
    "time_limit_a": [120, 900],
    "num_bag_folds_a": [5, 10],
    # "features": [
    #     ("purity", "initial_diameter", "tensile_strength", "temperature", "time"),
    #     # ("initial_diameter", "tensile_strength", "temperature", "time"),
    # ],
}

# 2. Data (same preparation as annealing_uts.ipynb, run once)

In [7]:
FILE_NAME = "dataset_annealing_tensile-strength.csv"

df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
df_float = proc.df_to_float(
    df_raw, drop_cols=["DOI", "is_Cu"], ignore_columns=["material"]
)
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"), "tensile_strength",
)
df = (
    df_with_masks[ALL_FEATURES + [TARGET]]
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

# No essay rows yet for UTS -> no is_essay column. The runr detects this
# and applies uniform weights (weight_on_essay_rows must stay 1.0).
print(f"Dataset: {df.shape}")

# Validation set: none held-out yet. When UTS essays arrive, build df_val
# from them (with the same columns) and pass df_val=df_val below.
df_val = None
df.head()

Dataset: (92, 6)


c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(
c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(


,purity,initial_diameter,tensile_strength,temperature,time,tensile_strength_final
0,99.99,23.0,1200.0,373.0,60.0,1180.0
1,99.99,23.0,1200.0,473.0,60.0,1150.0
2,99.99,23.0,1200.0,573.0,60.0,850.0
3,99.99,23.0,1200.0,673.0,60.0,600.0
4,99.99,23.0,1200.0,773.0,60.0,330.0


# 3. Base config

In [ ]:
base = ExperimentConfig(
    process=PROCESS,
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    y_max=None,
    y_min=0.0,
    presets_a = "best_quality",
)
print(base.run_id)

annealing-uts-v1-test__e3d4945f


# 4. Single run (sanity check before any grid)

Run the base config alone first; then repeat 2-3x with `tag="uts-v1-rep2"`
etc. to measure run-to-run noise (the floor below which grid differences
mean nothing).

In [ ]:
# result = runr.run_experiment(df, cfg=base, df_val=df_val)
# result["artifacts"]["metrics"]

2026-06-24 15:59:13,789 | INFO | src.modeling.Experiments | === Running annealing-uts-v1-test__e3d4945f ===
2026-06-24 15:59:13,790 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       12.44 GB / 31.57 GB (39.4%)
Disk Space Avail:   752.97 GB / 932.08 GB (80.8%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 120s


{'rmse': 40.78105998345382,
 'mae': 25.740681563667632,
 'mape': 8.221208254720606,
 'r2': 0.953848987214664}

# 5 Validation set

TBD

# 6. Grid

In [10]:
log = runr.run_grid(df, base_cfg=base, grid=GRID, df_val=df_val)
log

2026-06-24 16:02:00,853 | INFO | src.modeling.Experiments | Grid: 8 runs over ['time_limit_a', 'num_bag_folds_a', 'features']
2026-06-24 16:02:00,856 | INFO | src.modeling.Experiments | === Running annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__bbce27b1 ===
2026-06-24 16:02:00,856 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.63 GB / 31.57 GB (36.8%)
Disk Space Avail:   752.89 GB / 932.08 GB (80.8%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'we

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,r2,cov_0.5,cov_0.8,cov_0.9,cov_0.95,c_opt,pct_truncated_aleat,nnls_recovery_ok,mean_sigma_epist,mean_sigma_aleat
0,annealing-uts-v1-test__e3d4945f,2026-06-24T16:02:00,166.2,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v1-test\annealing-uts-v1-test__e3d4945f,annealing_uts,annealing-uts-v1-test,NaN,tensile_strength_final,...,0.95385,0.6413,0.8261,0.9239,0.9348,1.0902,52.17,True,23.18034,19.08715
1,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__bbce27b1,2026-06-24T16:04:01,120.9,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v1-test\annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__bbce27b1,annealing_uts,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time,annealing-uts-v1-test,tensile_strength_final,...,0.94854,0.4891,0.8152,0.8913,0.9348,1.0902,34.78,True,18.10781,19.08715
2,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=initial_diameter-tensile_strength-temperature-time__eced9448,2026-06-24T16:06:05,123.4,92,689947c5,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v1-test\annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=initial_diameter-tensile_strength-temperature-time__eced9448,annealing_uts,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=initial_diameter-tensile_strength-temperature-time,annealing-uts-v1-test,tensile_strength_final,...,0.96020,0.5000,0.7935,0.9022,0.9348,1.1431,30.43,True,18.42146,19.08715
3,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=purity-initial_diameter-tensile_strength-temperature-time__9c4b8022,2026-06-24T16:08:09,124.7,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v1-test\annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=purity-initial_diameter-tensile_strength-temperature-time__9c4b8022,annealing_uts,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=purity-initial_diameter-tensile_strength-temperature-time,annealing-uts-v1-test,tensile_strength_final,...,0.96463,0.5109,0.8370,0.9239,0.9565,1.0708,28.26,True,15.78590,19.08715
4,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=initial_diameter-tensile_strength-temperature-time__2bdb3be2,2026-06-24T16:10:33,144.0,92,689947c5,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v1-test\annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=initial_diameter-tensile_strength-temperature-time__2bdb3be2,annealing_uts,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=initial_diameter-tensile_strength-temperature-time,annealing-uts-v1-test,tensile_strength_final,...,0.96855,0.5000,0.8152,0.9022,0.9348,1.0502,32.61,True,15.89410,19.08715
5,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__e3d4945f,2026-06-24T16:13:06,152.9,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v1-test\annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__e3d4945f,annealing_uts,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time,annealing-uts-v1-test,tensile_strength_final,...,0.95385,0.6413,0.8261,0.9239,0.9348,1.0902,52.17,True,23.18034,19.08715
6,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a

# 7. Inspect results

In [11]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_num_bag_folds_a", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9",
    "c_opt", "pct_truncated_aleat", "mean_sigma_epist", "mean_sigma_aleat",
    "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_num_bag_folds_a,cfg_features,rmse,mae,cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
8,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=10__features=initial_diameter-tensile_strength-temperature-time__9b0ba1d3,120,10,initial_diameter|tensile_strength|temperature|time,32.28832,22.75341,0.9348,1.0902,54.35,21.55632,19.08715,172.9
7,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=10__features=purity-initial_diameter-tensile_strength-temperature-time__2c0d86a2,120,10,purity|initial_diameter|tensile_strength|temperature|time,33.54071,23.56783,0.9022,0.9296,46.74,21.92120,19.08715,175.7
4,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=initial_diameter-tensile_strength-temperature-time__2bdb3be2,60,10,initial_diameter|tensile_strength|temperature|time,33.66411,24.19113,0.9022,1.0502,32.61,15.89410,19.08715,144.0
3,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=10__features=purity-initial_diameter-tensile_strength-temperature-time__9c4b8022,60,10,purity|initial_diameter|tensile_strength|temperature|time,35.70117,24.57673,0.9239,1.0708,28.26,15.78590,19.08715,124.7
6,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=5__features=initial_diameter-tensile_strength-temperature-time__7c7745e2,120,5,initial_diameter|tensile_strength|temperature|time,37.74624,26.83952,0.9022,1.0902,42.39,21.39663,19.08715,150.3
2,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=initial_diameter-tensile_strength-temperature-time__eced9448,60,5,initial_diameter|tensile_strength|temperature|time,37.87000,27.36114,0.9022,1.1431,30.43,18.42146,19.08715,123.4
5,annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__e3d4945f,120,5,purity|initial_diameter|tensile_strength|temperature|time,40.78106,25.74068,0.9239,1.0902,52.17,23.18034,19.08715,152.9
0,annealing-uts-v1-test__e3d4945f,120,5,purity|initial_diameter|tensile_strength|temperature|time,40.78106,25.74068,0.9239,1.0902,52.17,23.18034,19.08715,166.2
1,annealing-uts-v1-test__time_limit_a=60__num_bag_folds_a=5__features=purity-initial_diameter-tensile_strength-temperature-time__bbce27b1,60,5,purity|initial_diameter|tensile_strength|temperature|time,43.06229,27.29911,0.8913,1.0902,34.78,18.10781,19.08715,120.9


In [13]:
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                  mae           cov_0.9          
                       mean       std       mean       std    mean       std
cfg_time_limit_a                                                            
60                37.574393  4.041603  25.857028  1.708443  0.9049  0.013669
120               37.027478  3.978459  24.928424  1.699537  0.9174  0.014572

# 8. Load a winner

In [ ]:
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-uts-v1-test__time_limit_a=120__num_bag_folds_a=10__features=initial_diameter-tensile_strength-temperature-time__9b0ba1d3


,alpha,empirical_coverage,gap
0,0.50,0.586957,0.086957
1,0.80,0.858696,0.058696
2,0.90,0.934783,0.034783
3,0.95,0.945652,-0.004348
